In [1]:

from pathlib import Path

import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "likely" /"tidytuesday" 

# Load local CSV files
absolute = pd.read_csv(
    DATA_DIR / "absolute_judgements.csv"
)

pairwise = pd.read_csv(
    DATA_DIR / "pairwise_comparisons.csv"
)

metadata = pd.read_csv(
    DATA_DIR / "respondent_metadata.csv"
)

# Inspect dataset dimensions
for name, df in {
    "absolute": absolute,
    "pairwise": pairwise,
    "metadata": metadata,
}.items():
    print(f"{name}: {df.shape}")


absolute: (98306, 4)
pairwise: (51740, 5)
metadata: (5174, 6)


In [2]:
first = (
    pairwise[pairwise["pair_id"] == 1]
    [["response_id", "term1", "term2", "selected"]]
    .rename(columns={
        "term1": "term1_first",
        "term2": "term2_first",
        "selected": "selected_first"
    })
)

last = (
    pairwise[pairwise["pair_id"] == 10]
    [["response_id", "term1", "term2", "selected"]]
    .rename(columns={
        "term1": "term1_last",
        "term2": "term2_last",
        "selected": "selected_last"
    })
)

repeat = first.merge(last, on="response_id")

repeat["same_pair"] = repeat.apply(
    lambda r: {r["term1_first"], r["term2_first"]}
    == {r["term1_last"], r["term2_last"]},
    axis=1
)

repeat["same_choice"] = (
    repeat["selected_first"] == repeat["selected_last"]
)

print("Same pair:", repeat["same_pair"].sum(), "/", len(repeat))

print(
    "Choice consistency:",
    repeat.loc[repeat["same_pair"], "same_choice"].mean()
)

Same pair: 5174 / 5174
Choice consistency: 0.9663703131039815


In [3]:
# Identify respondents who changed their choice
inconsistent = repeat.loc[
    repeat["same_pair"] & ~repeat["same_choice"]
].copy()

print("Inconsistent respondents:", len(inconsistent))

# Which repeated pair was shown?
inconsistent["pair"] = inconsistent.apply(
    lambda r: " vs ".join(sorted([
        r["term1_first"], r["term2_first"]
    ])),
    axis=1
)

print("\nInconsistency by repeated pair:")
display(
    inconsistent["pair"]
    .value_counts()
    .rename("inconsistent_responses")
)

Inconsistent respondents: 174

Inconsistency by repeated pair:


pair
Likely vs Probable                             23
Likely vs Very Good Chance                     18
Remote Chance vs Unlikely                       8
Almost No Chance vs Highly Unlikely             8
Improbable vs Unlikely                          7
About Even vs Might Happen                      6
Could Happen vs Unlikely                        6
Could Happen vs Might Happen                    5
Little Chance vs Unlikely                       4
Might Happen vs Realistic Possibility           4
Highly Unlikely vs Little Chance                4
May Happen vs Realistic Possibility             4
Almost Certain vs Will Happen                   4
Chances are Slight vs Improbable                4
Almost No Chance vs Unlikely                    3
Highly Unlikely vs Remote Chance                3
Highly Likely vs Highly Unlikely                3
Better than Even vs Very Good Chance            3
Highly Likely vs Improbable                     3
May Happen vs Might Happen                   

In [4]:
repeat["pair"] = repeat.apply(
    lambda r: " vs ".join(sorted([
        r["term1_first"], r["term2_first"]
    ])),
    axis=1
)

pair_consistency = (
    repeat.groupby("pair")
    .agg(
        respondents=("response_id", "count"),
        consistent=("same_choice", "sum")
    )
)

pair_consistency["consistency_rate"] = (
    pair_consistency["consistent"] /
    pair_consistency["respondents"]
)

display(
    pair_consistency.sort_values("consistency_rate")
)

,respondents,consistent,consistency_rate
pair,,,
Could Happen vs Might Happen,16,11,0.687500
About Even vs Might Happen,23,17,0.739130
Highly Unlikely vs Remote Chance,14,11,0.785714
Likely vs Probable,116,93,0.801724
Almost No Chance vs Unlikely,16,13,0.812500
...,...,...,...
Almost Certain vs Realistic Possibility,13,13,1.000000
Could Happen vs Likely,23,23,1.000000
Could Happen vs Improbable,19,19,1.000000


Do people agree on the numerical meaning of probability phrases, and does that agreement correspond to their pairwise choices?

In [5]:
# Join each pairwise comparison with the respondent's
# absolute probability estimates for both phrases.

absolute_wide = absolute.pivot(
    index="response_id",
    columns="term",
    values="probability"
)

pairwise_analysis = pairwise.copy()

pairwise_analysis["probability_term1"] = [
    absolute_wide.loc[rid, term]
    for rid, term in zip(
        pairwise_analysis["response_id"],
        pairwise_analysis["term1"]
    )
]

pairwise_analysis["probability_term2"] = [
    absolute_wide.loc[rid, term]
    for rid, term in zip(
        pairwise_analysis["response_id"],
        pairwise_analysis["term2"]
    )
]

pairwise_analysis["predicted_choice"] = np.where(
    pairwise_analysis["probability_term1"]
    > pairwise_analysis["probability_term2"],
    pairwise_analysis["term1"],
    np.where(
        pairwise_analysis["probability_term2"]
        > pairwise_analysis["probability_term1"],
        pairwise_analysis["term2"],
        "Tie"
    )
)

pairwise_analysis["choice_agreement"] = (
    pairwise_analysis["selected"]
    == pairwise_analysis["predicted_choice"]
)

display(pairwise_analysis.head())

,response_id,pair_id,term1,term2,selected,probability_term1,probability_term2,predicted_choice,choice_agreement
0,5177,1,Very Good Chance,Remote Chance,Very Good Chance,80,18,Very Good Chance,True
1,5177,2,Almost No Chance,Realistic Possibility,Realistic Possibility,9,68,Realistic Possibility,True
2,5177,3,Could Happen,Little Chance,Could Happen,66,20,Could Happen,True
3,5177,4,Chances are Slight,Improbable,Chances are Slight,27,10,Chances are Slight,True
4,5177,5,Probable,Unlikely,Probable,80,33,Probable,True


In [6]:
non_ties = pairwise_analysis[
    pairwise_analysis["predicted_choice"] != "Tie"
]

agreement_rate = non_ties["choice_agreement"].mean()

print(f"Agreement rate: {agreement_rate:.2%}")
print(f"Non-tied comparisons: {len(non_ties):,}")
print(
    "Tied comparisons:",
    (pairwise_analysis["predicted_choice"] == "Tie").sum()
)

Agreement rate: 92.46%
Non-tied comparisons: 49,379
Tied comparisons: 2361
